# 05 Extend Analysis Through 2025

Pull Google Trends through 2025, load the current Census workbook year sheets, and rerun the search-to-sales lag pipeline for 2018-2025.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from pytrends.request import TrendReq
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose


def find_project_root() -> Path:
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /Users/rad/Desktop/SearchLag


In [2]:
trends_path = RAW_DIR / "google_trends_2018_2025.csv"

searches = {
    "running_shoes": "running shoes",
    "furniture": "buy furniture",
    "laptops": "buy laptop",
    "apparel": "buy clothes",
    "washing_machines": "buy washing machine",
}

if trends_path.exists():
    trends_df = pd.read_csv(trends_path, index_col=0, parse_dates=True)
    print("Using existing Trends file:", trends_path)
    print(f"{len(trends_df)} weeks | {trends_df.index[0].date()} to {trends_df.index[-1].date()}")
else:
    pytrends = TrendReq(hl="en-US", tz=360)
    trends_full = {}

    for category, keyword in searches.items():
        print(f"Pulling: {keyword}")
        pytrends.build_payload(
            [keyword],
            timeframe="2018-01-01 2025-12-31",
            geo="US",
        )
        df = pytrends.interest_over_time()
        if not df.empty:
            trends_full[category] = df[keyword]
            print(f"  {len(df)} weeks | {df.index[0].date()} to {df.index[-1].date()}")
        else:
            print(f"  No data returned for {category}")
        time.sleep(3)

    trends_df = pd.DataFrame(trends_full)
    trends_df.index = pd.to_datetime(trends_df.index)
    trends_df.to_csv(trends_path)
    print("\nTrends saved:", trends_path)

Using existing Trends file: /Users/rad/Desktop/SearchLag/data/raw/google_trends_2018_2025.csv
96 weeks | 2018-01-01 to 2025-12-01


In [3]:
census_file = RAW_DIR / "census_retail.xlsx"

category_map = {
    "Clothing and clothing accessories stores": ["running_shoes", "apparel"],
    "Furniture and home furnishings stores": ["furniture"],
    "Electronics and appliance stores": ["laptops", "washing_machines"],
}

category_aliases = {
    "Clothing and clothing accessories stores": [
        "Clothing and clothing accessories stores",
        "Clothing and clothing access. stores",
    ],
    "Furniture and home furnishings stores": [
        "Furniture and home furnishings stores",
    ],
    "Electronics and appliance stores": [
        "Electronics and appliance stores",
    ],
}

alias_to_category = {
    alias: category
    for category, aliases in category_aliases.items()
    for alias in aliases
}

all_years = []

for year in ["2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]:
    try:
        raw = pd.read_excel(census_file, sheet_name=year, header=None)
        section_labels = raw.iloc[:, 1].astype(str).str.strip()
        not_adjusted_rows = section_labels[section_labels == "NOT ADJUSTED"].index
        adjusted_rows = section_labels[section_labels.str.match(r"^ADJUSTED")].index

        if len(not_adjusted_rows) == 0:
            raise ValueError("NOT ADJUSTED section not found")

        start_row = not_adjusted_rows[0] + 1
        adjusted_after_start = adjusted_rows[adjusted_rows > start_row]
        end_row = adjusted_after_start[0] if len(adjusted_after_start) else len(raw)
        month_labels = raw.iloc[4, 2:14].tolist()

        df = raw.iloc[start_row:end_row, [1, *range(2, 14)]].copy()
        df.columns = ["category", *month_labels]
        df["category"] = df["category"].astype(str).str.strip()
        df = df[df["category"].isin(alias_to_category)].copy()
        df["category"] = df["category"].map(alias_to_category)

        df_long = df.melt(
            id_vars=["category"],
            var_name="month_label",
            value_name="sales_millions",
        )
        df_long["date"] = pd.to_datetime(df_long["month_label"], errors="coerce")
        df_long["sales_millions"] = pd.to_numeric(df_long["sales_millions"], errors="coerce")
        df_long = df_long.dropna(subset=["date", "sales_millions"])
        all_years.append(df_long[["category", "date", "sales_millions"]])
        print(f"Year {year}: {len(df_long)} rows loaded")
    except Exception as e:
        print(f"Year {year} failed: {e}")

census_full = pd.concat(all_years, ignore_index=True)
census_full = census_full.sort_values("date")
census_path = PROCESSED_DIR / "census_clean_2018_2025.csv"
census_full.to_csv(census_path, index=False)
print(f"\nFull census data: {census_full['date'].min()} to {census_full['date'].max()}")
print(f"Total rows: {len(census_full)}")
print("Saved:", census_path)

Year 2018: 36 rows loaded
Year 2019: 36 rows loaded


/var/folders/2y/4bdg0r757s54_6cn9jgfy1rr0000gn/T/ipykernel_5961/44608129.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["date"] = pd.to_datetime(df_long["month_label"], errors="coerce")
/var/folders/2y/4bdg0r757s54_6cn9jgfy1rr0000gn/T/ipykernel_5961/44608129.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["date"] = pd.to_datetime(df_long["month_label"], errors="coerce")
/var/folders/2y/4bdg0r757s54_6cn9jgfy1rr0000gn/T/ipykernel_5961/44608129.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["date"] = pd.to_datetime(df_long["month_label"

Year 2020: 36 rows loaded
Year 2021: 36 rows loaded


Year 2022: 36 rows loaded
Year 2023: 36 rows loaded
Year 2024: 36 rows loaded


Year 2025: 36 rows loaded

Full census data: 2018-01-01 00:00:00 to 2025-12-01 00:00:00
Total rows: 288
Saved: /Users/rad/Desktop/SearchLag/data/processed/census_clean_2018_2025.csv


/var/folders/2y/4bdg0r757s54_6cn9jgfy1rr0000gn/T/ipykernel_5961/44608129.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["date"] = pd.to_datetime(df_long["month_label"], errors="coerce")
/var/folders/2y/4bdg0r757s54_6cn9jgfy1rr0000gn/T/ipykernel_5961/44608129.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["date"] = pd.to_datetime(df_long["month_label"], errors="coerce")
/var/folders/2y/4bdg0r757s54_6cn9jgfy1rr0000gn/T/ipykernel_5961/44608129.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["date"] = pd.to_datetime(df_long["month_label"

In [4]:
# Load both updated sources.
trends_df = pd.read_csv(
    RAW_DIR / "google_trends_2018_2025.csv",
    index_col=0,
    parse_dates=True,
)
census_df = pd.read_csv(
    PROCESSED_DIR / "census_clean_2018_2025.csv",
    parse_dates=["date"],
)

# Resample trends to monthly.
trends_monthly = trends_df.resample("MS").mean()

cat_map = {
    "running_shoes": "Clothing and clothing accessories stores",
    "apparel": "Clothing and clothing accessories stores",
    "furniture": "Furniture and home furnishings stores",
    "laptops": "Electronics and appliance stores",
    "washing_machines": "Electronics and appliance stores",
}

aligned_pairs = {}
for search_cat, census_cat in cat_map.items():
    search_s = trends_monthly[search_cat].copy()
    sales_s = census_df[
        census_df["category"] == census_cat
    ].set_index("date")["sales_millions"]

    combined = pd.DataFrame({
        "search": search_s,
        "sales": sales_s,
    }).dropna()

    aligned_pairs[search_cat] = combined
    print(
        f"{search_cat}: {len(combined)} months | "
        f"{combined.index[0].date()} to {combined.index[-1].date()}"
    )

cleaned_pairs = {}
for category, df in aligned_pairs.items():
    search_decomp = seasonal_decompose(
        df["search"],
        model="additive",
        period=12,
        extrapolate_trend="freq",
    )
    sales_decomp = seasonal_decompose(
        df["sales"],
        model="additive",
        period=12,
        extrapolate_trend="freq",
    )
    cleaned_pairs[category] = pd.DataFrame({
        "search_clean": search_decomp.trend + search_decomp.resid,
        "sales_clean": sales_decomp.trend + sales_decomp.resid,
    }).dropna()


def find_optimal_lag(df, max_lag=12):
    results = []
    for lag in range(0, max_lag + 1):
        if lag == 0:
            search = df["search_clean"].values
            sales = df["sales_clean"].values
        else:
            search = df["search_clean"].values[:-lag]
            sales = df["sales_clean"].values[lag:]

        min_len = min(len(search), len(sales))
        search = search[-min_len:]
        sales = sales[-min_len:]
        mask = ~(np.isnan(search) | np.isnan(sales))

        if mask.sum() < 8:
            continue

        corr, pvalue = stats.pearsonr(search[mask], sales[mask])
        n = mask.sum()
        stderr = 1.0 / np.sqrt(n - 3)
        delta = 1.96 * stderr
        corr_for_ci = np.clip(corr, -0.999999, 0.999999)

        results.append({
            "lag_months": lag,
            "correlation": round(corr, 4),
            "pvalue": round(pvalue, 6),
            "significant": pvalue < 0.05,
            "ci_lower": round(np.tanh(np.arctanh(corr_for_ci) - delta), 4),
            "ci_upper": round(np.tanh(np.arctanh(corr_for_ci) + delta), 4),
        })
    return pd.DataFrame(results)

summary_rows = []
print("\n=== UPDATED RESULTS (2018-2025) ===")

for category, df in cleaned_pairs.items():
    lag_df = find_optimal_lag(df)
    lag_df.to_csv(PROCESSED_DIR / f"{category}_lag_results_2018_2025.csv", index=False)
    sig = lag_df[lag_df["significant"]]

    if sig.empty:
        print(f"{category}: No significant lag found")
        summary_rows.append({
            "category": category,
            "optimal_lag": None,
            "peak_corr": None,
            "pvalue": None,
        })
        continue

    best = sig.loc[sig["correlation"].idxmax()]
    print(
        f"{category}: lag={int(best['lag_months'])} months | "
        f"r={best['correlation']} | p={best['pvalue']}"
    )

    summary_rows.append({
        "category": category,
        "optimal_lag": int(best["lag_months"]),
        "peak_corr": best["correlation"],
        "pvalue": best["pvalue"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = PROCESSED_DIR / "lag_summary_2018_2025.csv"
summary_df.to_csv(summary_path, index=False)
print("\n=== SUMMARY ===")
print(summary_df.to_string(index=False))
print("Saved:", summary_path)

running_shoes: 96 months | 2018-01-01 to 2025-12-01
apparel: 96 months | 2018-01-01 to 2025-12-01
furniture: 96 months | 2018-01-01 to 2025-12-01
laptops: 96 months | 2018-01-01 to 2025-12-01
washing_machines: 96 months | 2018-01-01 to 2025-12-01

=== UPDATED RESULTS (2018-2025) ===
running_shoes: lag=12 months | r=0.6406 | p=0.0
apparel: lag=11 months | r=-0.2147 | p=0.048431
furniture: lag=0 months | r=-0.2008 | p=0.049839
laptops: lag=5 months | r=-0.2407 | p=0.021561
washing_machines: No significant lag found

=== SUMMARY ===
        category  optimal_lag  peak_corr   pvalue
   running_shoes         12.0     0.6406 0.000000
         apparel         11.0    -0.2147 0.048431
       furniture          0.0    -0.2008 0.049839
         laptops          5.0    -0.2407 0.021561
washing_machines          NaN        NaN      NaN
Saved: /Users/rad/Desktop/SearchLag/data/processed/lag_summary_2018_2025.csv
